In [ ]:
import logging
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from ml_translate.data import get_dataloaders
from ml_translate.model import EncoderRNN, AttnDecoderRNN
from ml_translate.train import train
from ml_translate.eval import evaluate, evaluateRandomly

# Configure logging to display in notebook
logging.basicConfig(level=logging.INFO, format='%(message)s')

In [ ]:
hidden_size = 128
batch_size = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_lang, output_lang, train_dataloader, val_dataloader, test_dataloader, test_pairs = get_dataloaders(batch_size, device)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = AttnDecoderRNN(hidden_size, output_lang.n_words, device=device).to(device)

result = train(train_dataloader, encoder, decoder, 80, print_every=5, plot_every=5, val_dataloader=val_dataloader)


def showPlot(train_losses, val_losses=None):
    plt.figure()
    fig, ax = plt.subplots()
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(train_losses, label='Train')
    if val_losses:
        plt.plot(val_losses, label='Validation')
        plt.legend()
    plt.xlabel('Epochs (x5)')
    plt.ylabel('Loss')
    plt.title('Training Progress')


showPlot(result.train_losses, result.val_losses)

In [ ]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder, input_lang, output_lang, test_pairs, device)

In [ ]:
def showAttention(input_sentence, output_words, attentions):
    fig = plt.figure()
    ax = fig.add_subplot(111)
    cax = ax.matshow(attentions.cpu().numpy(), cmap="bone")
    fig.colorbar(cax)

    # Show label at every tick
    ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(1))

    # Set up axes
    ax.set_xticklabels([""] + input_sentence.split(" ") + ["<EOS>"], rotation=90)
    ax.set_yticklabels([""] + output_words)

    plt.show()


def evaluateAndShowAttention(input_sentence):
    output_words, attentions = evaluate(
        encoder, decoder, input_sentence, input_lang, output_lang, device
    )
    print("input =", input_sentence)
    print("output =", " ".join(output_words))
    showAttention(input_sentence, output_words, attentions[0, : len(output_words), :])


evaluateAndShowAttention("il n est pas aussi grand que son pere")
evaluateAndShowAttention("je suis trop fatigue pour conduire")
evaluateAndShowAttention("je suis desole si c est une question idiote")
evaluateAndShowAttention("je suis reellement fiere de vous")